# Omission 敏感性分析: Censor vs Drop

**项目**: GP-SPE 实验设计优化 — HDDM 参数提取管线

**文献依据**: Leng, X., Fengler, A., Shenhav, A., & Frank, M. J. (2025). The Perils of Omitting Omissions when Modeling Evidence Accumulation.

**分析目标**: 对比两种 Omission 处理方案对 DDM 参数估计的影响

| 方案 | 处理方式 | 描述 |
|:---|:---|:---|
| **Censor (右截尾)** | rt = T+W, response = 0 | 当前 HDDM pipeline 方案，告诉模型该试次在 deadline 前未穿越边界 |
| **Drop (直接丢弃)** | 删除 omission==1 行 | 仅保留有效试次，模型不知道 omission 的存在 |

> ⚠️ **注意**: Step 2a 和 Step 2b 的 HDDM 拟合需要在 **Docker 容器** (hcp4715/hddm) 中运行

---

## 工作流步骤

| Step | 内容 | 运行环境 |
|:---|:---|:---|
| Step 1 | 数据准备（生成 Drop + Censor 两套数据） | 本地 |
| Step 2a | Drop 方案 HDDM 拟合 | 🐳 Docker |
| Step 2b | Censor 方案 HDDM 拟合（可选，已有历史结果） | 🐳 Docker |
| Step 3 | 参数提取 + 定量比较 + 可视化 | 本地 |

---

## 环境设置

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from pathlib import Path
import re
import pickle
import os
import sys
import warnings
warnings.filterwarnings("ignore")

plt.rcParams["font.sans-serif"] = ["SimHei", "DejaVu Sans"]
plt.rcParams["axes.unicode_minus"] = False

BASE_DIR = Path(os.getcwd()).resolve()
if BASE_DIR.name in ("Omission", "Python_for_Check", "1_Code"):
    for _ in range({"Omission": 3, "Python_for_Check": 2, "1_Code": 1}.get(BASE_DIR.name, 0)):
        BASE_DIR = BASE_DIR.parent

print(f"项目根目录: {BASE_DIR}")

---

## Step 1: 数据准备

从已有的 HDDM_Ready 数据生成两套分析数据：
- **Drop 方案**: 删除 omission==1 的试次，仅保留有效试次
- **Censor 方案**: 保留原有格式（omission 试次 rt=deadline, response=0）

In [ ]:
HDDM_READY_DIR = BASE_DIR / "2_Data" / "Real_Data" / "HDDM_Ready"
OUT_DIR = BASE_DIR / "2_Data" / "Generate_Data" / "Omission_Sensitivity"
DROP_DIR = OUT_DIR / "drop_omission"
CENSOR_DIR = OUT_DIR / "censored"

DROP_DIR.mkdir(parents=True, exist_ok=True)
CENSOR_DIR.mkdir(parents=True, exist_ok=True)

csv_files = sorted(HDDM_READY_DIR.glob("hddm_data_group*.csv"))
print(f"发现 {len(csv_files)} 个 HDDM 就绪数据文件")
for f in csv_files:
    print(f"  {f.name}")

In [ ]:
summary_rows = []

for csv_path in csv_files:
    fname = csv_path.stem
    df = pd.read_csv(csv_path)

    n_total = len(df)
    n_omission = int(df["omission"].sum())
    n_valid = n_total - n_omission
    omission_rate = n_omission / n_total * 100

    match = re.search(r"group(\d+)_P(\d+)_T(\d+)_W(\d+)", fname)
    if not match:
        print(f"  ⚠️ 无法解析文件名: {fname}")
        continue
    group_id = int(match.group(1))
    P_val = int(match.group(2))
    T_val = int(match.group(3))
    W_val = int(match.group(4))

    df_drop = df[df["omission"] == 0].copy()
    df_drop = df_drop.drop(columns=["omission"])
    df_drop.to_csv(DROP_DIR / f"{fname}_drop.csv", index=False)

    df.to_csv(CENSOR_DIR / f"{fname}_censor.csv", index=False)

    n_subj = df["subj_idx"].nunique()
    d_acc = df_drop["response"].mean() if len(df_drop) > 0 else np.nan

    summary_rows.append({
        "group_id": group_id, "P": P_val, "T_ms": T_val, "W_ms": W_val,
        "M_ms": T_val + W_val, "n_subjects": n_subj,
        "n_total_trials": n_total, "n_omission": n_omission,
        "omission_rate": omission_rate, "n_drop_trials": len(df_drop),
        "n_censor_trials": n_total, "drop_acc": d_acc,
    })

    print(f"  Group {group_id}: P={P_val}, T={T_val}ms, W={W_val}ms")
    print(f"    被试={n_subj} | 总试次={n_total} | 遗漏={n_omission} ({omission_rate:.1f}%)")
    print(f"    Drop={len(df_drop)}, Censor={n_total}, 正确率={d_acc:.3f}")

summary_df = pd.DataFrame(summary_rows)
summary_df.to_csv(OUT_DIR / "data_summary.csv", index=False)
print(f"\n✅ 数据准备完成! 汇总表 → {OUT_DIR / 'data_summary.csv'}")

In [ ]:
summary_df[["group_id", "P", "T_ms", "W_ms", "omission_rate", "n_drop_trials", "n_censor_trials", "drop_acc"]]

---

## Step 2a: Drop 方案 HDDM 拟合

> 🐳 **此单元格在 Docker 容器中运行**

删除所有 omission 试次后，对仅包含有效试次的数据进行 DDM 拟合。

Docker 启动命令参考：
```bash
docker run -it --rm --cpus=4 -v \
  /d/GitHub_programe/GitHub/Guassion-Process-Experiment-Design:/home/jovyan/work \
  -p 8888:8888 hcp4715/hddm jupyter notebook
```

In [ ]:
try:
    import hddm
    print("✅ hddm 已安装")
except ImportError:
    print("❌ hddm 未安装！请确认当前在 Docker 容器 (hcp4715/hddm) 中运行")
    print("如果已在 Docker 中，请检查 kernel 是否正确")
    sys.exit(1)

DROP_DATA_DIR = BASE_DIR / "2_Data" / "Generate_Data" / "Omission_Sensitivity" / "drop_omission"
DROP_OUT_DIR = BASE_DIR / "2_Data" / "Generate_Data" / "Omission_Sensitivity" / "drop_traces"
DROP_OUT_DIR.mkdir(parents=True, exist_ok=True)

drop_csv_files = sorted(DROP_DATA_DIR.glob("hddm_data_group*_drop.csv"))
print(f"发现 {len(drop_csv_files)} 个 Drop 数据文件")

In [ ]:
for csv_path in drop_csv_files:
    fname = csv_path.stem
    print(f"\n{'=' * 50}")
    print(f"拟合 [DROP]: {fname}")
    print(f"{'=' * 50}")

    df = pd.read_csv(csv_path)
    n_subj = df["subj_idx"].nunique()
    n_trials = len(df)
    n_self = (df["identity"] == 1).sum()
    n_stranger = (df["identity"] == 0).sum()
    acc = df["response"].mean()

    print(f"  被试数: {n_subj}, 试次数: {n_trials}")
    print(f"  Self: {n_self}, Stranger: {n_stranger}")
    print(f"  正确率: {acc:.3f}")

    model = hddm.HDDM(
        df,
        depends_on={"v": "identity"},
        include=["v", "a", "t", "z"],
        bias=False,
        p_outlier=0.05,
    )

    print("  开始 MCMC 采样 (3000 draws, 500 burn)...")
    db_name = f"traces_drop_{fname}.db"
    model.sample(3000, burn=500, dbname=db_name, db="pickle")
    print("  ✅ 采样完成")

    stats = model.gen_stats()
    stats.to_csv(DROP_OUT_DIR / f"{fname}_stats.csv")

    try:
        traces_raw = model.get_traces()
        traces_simple = {}
        for key, val in traces_raw.items():
            try:
                arr = np.asarray(val, dtype=float).flatten()
                if len(arr) > 0:
                    traces_simple[key] = arr
            except Exception:
                continue
        if traces_simple:
            with open(DROP_OUT_DIR / f"{fname}_traces.pkl", "wb") as f:
                pickle.dump(traces_simple, f, protocol=pickle.HIGHEST_PROTOCOL)
            np.savez_compressed(DROP_OUT_DIR / f"{fname}_traces.npz", **traces_simple)
            print(f"  迹线已保存 ({len(traces_simple)} 个参数)")
    except Exception as e:
        print(f"  ⚠️ 迹线保存失败: {e}")

    try:
        os.remove(db_name)
    except Exception:
        pass

print(f"\n✅ Drop 方案拟合完成! 结果保存于: {DROP_OUT_DIR}")

---

## Step 2b: Censor 方案 HDDM 拟合（可选）

> 🐳 **此单元格在 Docker 容器中运行**
>
> ⚠️ 如果已有历史 HDDM_Traces 结果（位于 `2_Data/Real_Data/HDDM_Traces/`），可跳过此步。此处提供完整重跑选项以保证两种方案使用完全相同的 MCMC 配置。

In [ ]:
CENSOR_DATA_DIR = BASE_DIR / "2_Data" / "Generate_Data" / "Omission_Sensitivity" / "censored"
CENSOR_OUT_DIR = BASE_DIR / "2_Data" / "Generate_Data" / "Omission_Sensitivity" / "censor_traces"
CENSOR_OUT_DIR.mkdir(parents=True, exist_ok=True)

censor_csv_files = sorted(CENSOR_DATA_DIR.glob("hddm_data_group*_censor.csv"))
print(f"发现 {len(censor_csv_files)} 个 Censor 数据文件")

In [ ]:
for csv_path in censor_csv_files:
    fname = csv_path.stem
    print(f"\n{'=' * 50}")
    print(f"拟合 [CENSOR]: {fname}")
    print(f"{'=' * 50}")

    df = pd.read_csv(csv_path)
    n_subj = df["subj_idx"].nunique()
    n_trials = len(df)
    n_omission = df["omission"].sum() if "omission" in df.columns else 0
    omission_rate = n_omission / n_trials * 100

    print(f"  被试数: {n_subj}, 试次数: {n_trials}")
    print(f"  遗漏试次: {int(n_omission)} ({omission_rate:.1f}%)")

    model = hddm.HDDM(
        df,
        depends_on={"v": "identity"},
        include=["v", "a", "t", "z"],
        bias=False,
        p_outlier=0.05,
    )

    print("  开始 MCMC 采样 (3000 draws, 500 burn)...")
    db_name = f"traces_censor_{fname}.db"
    model.sample(3000, burn=500, dbname=db_name, db="pickle")
    print("  ✅ 采样完成")

    stats = model.gen_stats()
    stats.to_csv(CENSOR_OUT_DIR / f"{fname}_stats.csv")

    try:
        traces_raw = model.get_traces()
        traces_simple = {}
        for key, val in traces_raw.items():
            try:
                arr = np.asarray(val, dtype=float).flatten()
                if len(arr) > 0:
                    traces_simple[key] = arr
            except Exception:
                continue
        if traces_simple:
            with open(CENSOR_OUT_DIR / f"{fname}_traces.pkl", "wb") as f:
                pickle.dump(traces_simple, f, protocol=pickle.HIGHEST_PROTOCOL)
            np.savez_compressed(CENSOR_OUT_DIR / f"{fname}_traces.npz", **traces_simple)
            print(f"  迹线已保存 ({len(traces_simple)} 个参数)")
    except Exception as e:
        print(f"  ⚠️ 迹线保存失败: {e}")

    try:
        os.remove(db_name)
    except Exception:
        pass

print(f"\n✅ Censor 方案拟合完成! 结果保存于: {CENSOR_OUT_DIR}")

---

## Step 3: 参数提取与比较分析

> 📍 **此单元格在本地运行**

从 HDDM 拟合的 stats 文件中提取各组 DDM 参数的后验均值和 95% CI，对比 Censor vs Drop 两种方案。

In [ ]:
def load_params_from_stats(stats_dir, prefix_pattern):
    """从 stats 文件提取 DDM 参数"""
    stats_files = sorted(Path(stats_dir).glob("*_stats.csv"))
    if not stats_files:
        return None

    all_params = []
    for stats_path in stats_files:
        fname = stats_path.stem.replace("_stats", "")

        match_dict = {}
        for prefix in prefix_pattern:
            m = re.search(prefix, fname)
            if m:
                match_dict.update(m.groupdict())
        if not match_dict:
            continue

        stats = pd.read_csv(stats_path, index_col=0)

        param_map = {
            "v(0)": "v_stranger", "v(1)": "v_self",
            "a": "a", "t": "t", "z": "z",
        }

        row = {}
        for k, v in match_dict.items():
            try:
                row[k] = int(v)
            except ValueError:
                row[k] = v

        for stat_key, label in param_map.items():
            if stat_key in stats.index:
                row[f"{label}_mean"] = float(stats.loc[stat_key, "mean"])
                row[f"{label}_std"] = float(stats.loc[stat_key, "std"])
                row[f"{label}_q025"] = float(stats.loc[stat_key, "2.5q"])
                row[f"{label}_q975"] = float(stats.loc[stat_key, "97.5q"])

        if "v_self_mean" in row and "v_stranger_mean" in row:
            row["SPE_v"] = row["v_self_mean"] - row["v_stranger_mean"]

        all_params.append(row)

    if not all_params:
        return None
    return pd.DataFrame(all_params).sort_values("group_id")

In [ ]:
CENSOR_TRACE_DIR_NEW = BASE_DIR / "2_Data" / "Generate_Data" / "Omission_Sensitivity" / "censor_traces"
CENSOR_TRACE_DIR_OLD = BASE_DIR / "2_Data" / "Real_Data" / "HDDM_Traces"
DROP_TRACE_DIR = BASE_DIR / "2_Data" / "Generate_Data" / "Omission_Sensitivity" / "drop_traces"
DATA_SUMMARY_PATH = BASE_DIR / "2_Data" / "Generate_Data" / "Omission_Sensitivity" / "data_summary.csv"
FIG_DIR = BASE_DIR / "3_Figures" / "Omission_Sensitivity"
FIG_DIR.mkdir(parents=True, exist_ok=True)

data_summary = pd.read_csv(DATA_SUMMARY_PATH)

print("加载 Censor 方案参数...")
censor_patterns = {
    str(CENSOR_TRACE_DIR_NEW): [r"hddm_data_group(?P<group_id>\d+)_P(?P<P>\d+)_T(?P<T_ms>\d+)_W(?P<W_ms>\d+)_censor"],
    str(CENSOR_TRACE_DIR_OLD): [r"hddm_data_group(?P<group_id>\d+)_P(?P<P>\d+)_T(?P<T_ms>\d+)_W(?P<W_ms>\d+)"],
}

censor_df = None
censor_source = None
for dir_path, patterns in censor_patterns.items():
    censor_df = load_params_from_stats(Path(dir_path), patterns)
    if censor_df is not None:
        censor_source = dir_path
        print(f"  从 {dir_path} 加载 {len(censor_df)} 组 Censor 参数")
        break

if censor_df is None:
    raise FileNotFoundError("未找到 Censor 方案结果！请先运行 Step 2b (Docker)")

censor_df["M_ms"] = censor_df["T_ms"] + censor_df["W_ms"]

print("\n加载 Drop 方案参数...")
drop_df = load_params_from_stats(
    DROP_TRACE_DIR,
    [r"hddm_data_group(?P<group_id>\d+)_P(?P<P>\d+)_T(?P<T_ms>\d+)_W(?P<W_ms>\d+)_drop"],
)

if drop_df is not None:
    drop_df["M_ms"] = drop_df["T_ms"] + drop_df["W_ms"]
    print(f"  加载 {len(drop_df)} 组 Drop 参数")
else:
    print("  ⚠️ 未找到 Drop 方案结果！请先运行 Step 2a (Docker)")

In [ ]:
PARAM_KEYS = ["v_self", "v_stranger", "SPE_v", "a", "t", "z"]
PARAM_LABELS = {
    "v_self": "Drift Rate v_self",
    "v_stranger": "Drift Rate v_stranger",
    "SPE_v": "SPE in Drift Rate (v_self - v_stranger)",
    "a": "Decision Boundary a",
    "t": "Nondecision Time t (s)",
    "z": "Starting Point z",
}

cmp_data = censor_df.merge(
    data_summary[["group_id", "omission_rate", "n_omission", "n_total_trials", "n_drop_trials"]],
    on="group_id", how="left",
)

if drop_df is not None:
    cmp_data = cmp_data.merge(
        drop_df, on=["group_id", "P", "T_ms", "W_ms", "M_ms"],
        how="inner", suffixes=("_censor", "_drop"),
    )

    print("\n" + "=" * 70)
    print("Omission 处理方法对比: Censor vs Drop")
    print("=" * 70)
    for param in PARAM_KEYS:
        pdata = []
        for _, row in cmp_data.iterrows():
            c_mean = row.get(f"{param}_mean_censor", np.nan)
            d_mean = row.get(f"{param}_mean_drop", np.nan)
            c_lo = row.get(f"{param}_q025_censor", np.nan)
            c_hi = row.get(f"{param}_q975_censor", np.nan)
            d_lo = row.get(f"{param}_q025_drop", np.nan)
            d_hi = row.get(f"{param}_q975_drop", np.nan)
            delta = d_mean - c_mean
            ci_overlap = "YES" if not ((c_hi < d_lo) or (d_hi < c_lo)) else "NO ***"
            pdata.append({
                "Group": f"G{row['group_id']:.0f}",
                "Censor": f"{c_mean:.3f}",
                "Drop": f"{d_mean:.3f}",
                "Delta": f"{delta:.3f}",
                "|Δ|>0.5?": "YES" if abs(delta) > 0.5 else "",
                "95%CI Overlap": ci_overlap,
            })
        print(f"\n--- {PARAM_LABELS[param]} ---")
        print(pd.DataFrame(pdata).to_string(index=False))
else:
    print("\n⚠️ Drop 结果缺失，跳过定量比较")

In [ ]:
if drop_df is not None:
    comparison_rows = []
    for _, row in cmp_data.iterrows():
        for key in PARAM_KEYS:
            c_mean = row.get(f"{key}_mean_censor", np.nan)
            d_mean = row.get(f"{key}_mean_drop", np.nan)
            c_std = row.get(f"{key}_std_censor", np.nan)
            d_std = row.get(f"{key}_std_drop", np.nan)
            c_lo = row.get(f"{key}_q025_censor", np.nan)
            c_hi = row.get(f"{key}_q975_censor", np.nan)
            d_lo = row.get(f"{key}_q025_drop", np.nan)
            d_hi = row.get(f"{key}_q975_drop", np.nan)
            delta = d_mean - c_mean
            pooled_se = np.sqrt(c_std**2 + d_std**2) if not np.isnan(c_std) and not np.isnan(d_std) else np.nan
            cohens_d = delta / pooled_se if pooled_se and not np.isnan(pooled_se) and pooled_se > 0 else np.nan
            ci_overlap = not ((c_hi < d_lo) or (d_hi < c_lo))
            comparison_rows.append({
                "group_id": row["group_id"], "parameter": key,
                "censor_mean": c_mean, "censor_q025": c_lo, "censor_q975": c_hi,
                "drop_mean": d_mean, "drop_q025": d_lo, "drop_q975": d_hi,
                "delta": delta, "abs_delta": abs(delta),
                "ci_overlap": ci_overlap, "cohens_d": cohens_d,
            })

    comp_df = pd.DataFrame(comparison_rows)
    comp_df.to_csv(BASE_DIR / "2_Data" / "Generate_Data" / "Omission_Sensitivity" / "sensitivity_comparison.csv", index=False)

    print("Cohen's d 汇总 (Drop - Censor):")
    for param in PARAM_KEYS:
        d_vals = comp_df[comp_df["parameter"] == param]["cohens_d"].dropna()
        if len(d_vals) > 0:
            print(f"  {PARAM_LABELS[param]}: mean d={d_vals.mean():.3f}, max |d|={d_vals.abs().max():.3f}")

---

## Step 3a: 可视化 — 参数对比柱状图

Censor vs Drop 方案下 6 个 DDM 参数的后验均值与 95% CI 对比。

In [ ]:
if drop_df is not None:
    fig, axes = plt.subplots(2, 3, figsize=(18, 12))
    axes = axes.flatten()

    for i, key in enumerate(PARAM_KEYS):
        ax = axes[i]
        c_mean = cmp_data[f"{key}_mean_censor"].values
        d_mean = cmp_data[f"{key}_mean_drop"].values
        c_lo = cmp_data[f"{key}_q025_censor"].values
        c_hi = cmp_data[f"{key}_q975_censor"].values
        d_lo = cmp_data[f"{key}_q025_drop"].values
        d_hi = cmp_data[f"{key}_q975_drop"].values
        groups = cmp_data["group_id"].values

        x = np.arange(len(groups))
        width = 0.35

        ax.bar(x - width / 2, c_mean, width,
               yerr=[c_mean - c_lo, c_hi - c_mean],
               label="Censor (rt=deadline)", color="#4472C4", alpha=0.85,
               capsize=4, error_kw={"linewidth": 1.5})
        ax.bar(x + width / 2, d_mean, width,
               yerr=[d_mean - d_lo, d_hi - d_mean],
               label="Drop (remove omission)", color="#ED7D31", alpha=0.85,
               capsize=4, error_kw={"linewidth": 1.5})

        ax.set_xticks(x)
        ax.set_xticklabels([f"G{g:.0f}" for g in groups], fontsize=9)
        ax.set_ylabel(PARAM_LABELS[key], fontsize=10)
        ax.axhline(y=0, color="gray", linestyle="--", linewidth=0.8)
        ax.legend(fontsize=8, loc="best")
        ax.grid(axis="y", alpha=0.3)

    fig.suptitle("Omission 敏感性分析: Censor vs Drop 方案下 DDM 参数比较",
                 fontsize=14, fontweight="bold")
    plt.tight_layout()
    fig_path = FIG_DIR / "sensitivity_censor_vs_drop_params.png"
    plt.savefig(fig_path, dpi=200, bbox_inches="tight")
    plt.show()
    print(f"参数对比图 → {fig_path}")
else:
    print("⚠️ 跳过：需要 Drop 方案拟合结果")

---

## Step 3b: 可视化 — 一致性散点图

横轴为 Censor 方案参数，纵轴为 Drop 方案参数。对角线表示完全一致。颜色表示遗漏率。

In [ ]:
if drop_df is not None:
    fig2, axes2 = plt.subplots(2, 3, figsize=(18, 12))
    axes2 = axes2.flatten()

    for i, key in enumerate(PARAM_KEYS):
        ax = axes2[i]
        c_mean = cmp_data[f"{key}_mean_censor"].values
        d_mean = cmp_data[f"{key}_mean_drop"].values
        omission_rates = cmp_data["omission_rate"].values
        groups = cmp_data["group_id"].values

        scatter = ax.scatter(c_mean, d_mean, c=omission_rates, s=100,
                             cmap="RdYlGn_r", edgecolors="black", linewidth=0.8,
                             vmin=0, vmax=80)

        for j, gid in enumerate(groups):
            ax.annotate(f"G{gid:.0f}", (c_mean[j], d_mean[j]),
                        textcoords="offset points", xytext=(6, 6), fontsize=7, alpha=0.8)

        all_vals = np.concatenate([c_mean, d_mean])
        vmin, vmax = all_vals.min(), all_vals.max()
        pad = (vmax - vmin) * 0.1
        vmin -= pad
        vmax += pad
        ax.plot([vmin, vmax], [vmin, vmax], "k--", linewidth=0.8, alpha=0.5)
        ax.set_xlim(vmin, vmax)
        ax.set_ylim(vmin, vmax)
        ax.set_xlabel(f"Censor {key}", fontsize=9)
        ax.set_ylabel(f"Drop {key}", fontsize=9)
        ax.set_title(PARAM_LABELS[key], fontsize=10)
        ax.set_aspect("equal")
        ax.grid(alpha=0.3)

    cbar = fig2.colorbar(scatter, ax=axes2, orientation="vertical",
                          fraction=0.02, pad=0.04)
    cbar.set_label("Omission Rate (%)", fontsize=9)

    fig2.suptitle("Censor vs Drop 一致性散点图 (颜色=遗漏率)", fontsize=14, fontweight="bold")
    plt.tight_layout()
    fig2_path = FIG_DIR / "sensitivity_scatter_censor_vs_drop.png"
    plt.savefig(fig2_path, dpi=200, bbox_inches="tight")
    plt.show()
    print(f"一致性散点图 → {fig2_path}")
else:
    print("⚠️ 跳过：需要 Drop 方案拟合结果")

---

## Step 3c: 可视化 — 参数差异 Δ = Drop − Censor

橙色柱表示 |Δ| > 0.5，提示该组参数在两个方案间有实质性差异。

In [ ]:
if drop_df is not None:
    fig3, axes3 = plt.subplots(2, 3, figsize=(18, 12))
    axes3 = axes3.flatten()

    for i, key in enumerate(PARAM_KEYS):
        ax = axes3[i]
        delta_vals = cmp_data[f"{key}_mean_drop"].values - cmp_data[f"{key}_mean_censor"].values
        omission_rates = cmp_data["omission_rate"].values
        groups = cmp_data["group_id"].values

        colors_delta = ["#4472C4" if abs(d) < 0.5 else "#ED7D31" for d in delta_vals]
        ax.bar(np.arange(len(groups)), delta_vals, color=colors_delta, alpha=0.85,
               edgecolor="black", linewidth=0.5)
        ax.axhline(y=0, color="black", linewidth=0.8)
        ax.set_xticks(np.arange(len(groups)))
        ax.set_xticklabels([f"G{g:.0f}\n{or_:.0f}%" for g, or_ in zip(groups, omission_rates)], fontsize=8)
        ax.set_ylabel(f"Δ {key}", fontsize=10)
        ax.set_title(f"{PARAM_LABELS[key]} (Drop - Censor)", fontsize=10)
        ax.grid(axis="y", alpha=0.3)

    fig3.suptitle("参数差异 Δ = Drop - Censor (橙色=|Δ|>0.5)", fontsize=14, fontweight="bold")
    plt.tight_layout()
    fig3_path = FIG_DIR / "sensitivity_delta_params.png"
    plt.savefig(fig3_path, dpi=200, bbox_inches="tight")
    plt.show()
    print(f"参数差异图 → {fig3_path}")
else:
    print("⚠️ 跳过：需要 Drop 方案拟合结果")

---

## Step 3d: 可视化 — Δ 与遗漏率的关系

参数差异是否随遗漏率升高而增大？红色虚线为线性趋势线。

In [ ]:
if drop_df is not None:
    fig4, axes4 = plt.subplots(2, 3, figsize=(18, 12))
    axes4 = axes4.flatten()

    for i, key in enumerate(PARAM_KEYS):
        ax = axes4[i]
        delta_vals = cmp_data[f"{key}_mean_drop"].values - cmp_data[f"{key}_mean_censor"].values
        omission_rates = cmp_data["omission_rate"].values
        groups = cmp_data["group_id"].values

        ax.scatter(omission_rates, delta_vals, s=120, c="#4472C4",
                   edgecolors="black", linewidth=0.8, alpha=0.85)

        for j, gid in enumerate(groups):
            ax.annotate(f"G{gid:.0f}", (omission_rates[j], delta_vals[j]),
                        textcoords="offset points", xytext=(5, 5), fontsize=8)

        ax.axhline(y=0, color="gray", linestyle="--", linewidth=0.8, alpha=0.5)

        if len(omission_rates) >= 2:
            z = np.polyfit(omission_rates, delta_vals, 1)
            p = np.poly1d(z)
            x_line = np.linspace(min(omission_rates), max(omission_rates), 100)
            ax.plot(x_line, p(x_line), "r--", linewidth=1, alpha=0.5,
                    label=f"Trend (slope={z[0]:.4f})")
            ax.legend(fontsize=7)

        ax.set_xlabel("Omission Rate (%)", fontsize=9)
        ax.set_ylabel(f"Δ {key}", fontsize=10)
        ax.set_title(PARAM_LABELS[key], fontsize=10)
        ax.grid(alpha=0.3)

    fig4.suptitle("参数差异与遗漏率的关系", fontsize=14, fontweight="bold")
    plt.tight_layout()
    fig4_path = FIG_DIR / "sensitivity_delta_vs_omission_rate.png"
    plt.savefig(fig4_path, dpi=200, bbox_inches="tight")
    plt.show()
    print(f"遗漏率关系图 → {fig4_path}")
else:
    print("⚠️ 跳过：需要 Drop 方案拟合结果")

---

## Step 3e: 可视化 — 遗漏率与试次分布总览

各组 Omission 率及两种方案下保留/丢弃的试次数对比。

In [ ]:
fig5, ax5 = plt.subplots(figsize=(12, 5))

groups = cmp_data["group_id"].values.astype(int)
n_total = cmp_data["n_total_trials"].values
n_drop = cmp_data["n_drop_trials"].values
n_omission_list = cmp_data["n_omission"].values
omission_rates = cmp_data["omission_rate"].values

x = np.arange(len(groups))
width = 0.4

ax5.bar(x - width/2, n_drop, width, label="保留试次 (Drop方案)", color="#4472C4", alpha=0.85)
ax5.bar(x - width/2, n_omission_list, width, bottom=n_drop,
        label="丢弃试次 (Omission)", color="#D62728", alpha=0.65)
ax5.bar(x + width/2, n_total, width, label="总试次 (Censor方案)", color="#7F7F7F", alpha=0.85)

for i, (k, o) in enumerate(zip(n_drop, n_omission_list)):
    total = k + o
    ax5.text(i - width/2, total + 30, f"{o/total*100:.0f}%", ha="center", fontsize=8, fontweight="bold")

ax5.set_xticks(x)
ax5.set_xticklabels([f"G{g}\nP={r['P']:.0f},T={r['T_ms']:.0f},W={r['W_ms']:.0f}"
                     for g, (_, r) in zip(groups, cmp_data.iterrows())], fontsize=8)
ax5.set_ylabel("试次数", fontsize=11)
ax5.set_title("各组 Omission 率与试次分布", fontsize=13, fontweight="bold")
ax5.legend(fontsize=9, loc="upper right")
ax5.grid(axis="y", alpha=0.3)

plt.tight_layout()
fig5_path = FIG_DIR / "omission_summary_by_group.png"
plt.savefig(fig5_path, dpi=200, bbox_inches="tight")
plt.show()
print(f"遗漏率汇总图 → {fig5_path}")

---

## Step 3f: 结论速览

In [ ]:
if drop_df is not None:
    print("=" * 60)
    print("关键结论速览: Censor vs Drop 方案参数差异")
    print("=" * 60)
    for key, label in PARAM_LABELS.items():
        deltas = cmp_data[f"{key}_mean_drop"].values - cmp_data[f"{key}_mean_censor"].values
        mean_d = np.mean(deltas)
        max_d = np.max(np.abs(deltas))
        sig_count = np.sum(np.abs(deltas) > 0.5)
        print(f"  {label}:")
        print(f"    平均Δ={mean_d:.4f}, 最大|Δ|={max_d:.4f}, |Δ|>0.5的组数={sig_count}/{len(deltas)}")
else:
    print("⚠️ 结论速览需要 Drop 方案拟合结果。请先在 Docker 中运行 Step 2a。")

---

## 输出文件汇总

| 文件 | 位置 |
|:---|:---|
| 数据汇总表 | `2_Data/Generate_Data/Omission_Sensitivity/data_summary.csv` |
| Drop 方案数据 | `2_Data/Generate_Data/Omission_Sensitivity/drop_omission/` |
| Censor 方案数据 | `2_Data/Generate_Data/Omission_Sensitivity/censored/` |
| Drop 方案 HDDM 结果 | `2_Data/Generate_Data/Omission_Sensitivity/drop_traces/` |
| Censor 方案 HDDM 结果 | `2_Data/Generate_Data/Omission_Sensitivity/censor_traces/` 或 `2_Data/Real_Data/HDDM_Traces/` |
| 参数比较表 | `2_Data/Generate_Data/Omission_Sensitivity/sensitivity_comparison.csv` |
| 图表输出 | `3_Figures/Omission_Sensitivity/` |